# Module 5: SCD Type 2 Implementation

**Objective**: Implement Slowly Changing Dimension Type 2 for customer dimension tracking.

## Key Concepts
- Surrogate keys vs business keys
- Effective/expiration dates
- is_current flag
- MERGE logic in PySpark

In [ ]:
# ── SparkSession: Databricks Connect (remote) / Local fallback ──
from pathlib import Path

try:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()
    MODE = 'databricks'
    S3_RAW = "s3a://sparkling-data-test/data/raw"
    print(f"✅ Databricks Connect | Spark {spark.version}")
except Exception:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.appName("Module05-SCD").master("local[*]").config("spark.sql.shuffle.partitions", "8").getOrCreate()
    MODE = 'local'
    S3_RAW = None
    print(f"✅ Local Spark {spark.version} | UI: http://localhost:4040")

DATA_RAW = Path("../data/raw")  # local CSV fallback path
print(f"Mode: {MODE}")

In [ ]:
# ── Load data (S3 Parquet or local CSV) ──
customers_df = spark.read.parquet(f"{S3_RAW}/customers") if MODE == "databricks" else spark.read.csv(str(DATA_RAW / "customers.csv"), header=True, inferSchema=True)

customers_df.show(5)

## 1. Initialize SCD Table

In [ ]:
# Create initial SCD table with versioning columns
dim_customer_init = customers_df.select(
    monotonically_increasing_id().alias("customer_key"),  # Surrogate key
    col("customer_id"),  # Business key
    col("name"),
    col("email"),
    col("segment"),
    col("kyc_status"),
    current_date().alias("effective_date"),
    lit("9999-12-31").cast("date").alias("expiration_date"),
    lit(True).alias("is_current")
)
dim_customer_init.show(5)

## 2. Simulate Updates (New Data)

In [ ]:
# Simulate some customer changes
updated_customers = customers_df.limit(100).withColumn("segment", lit("VIP Updated"))
print(f"Simulated {updated_customers.count()} updated customers")
updated_customers.show(3)

## 3. SCD Type 2 MERGE Logic

In [ ]:
def apply_scd2(existing_df, updates_df, key_col, track_cols):
    """Apply SCD Type 2 logic."""
    
    # Find changed records
    current_records = existing_df.filter(col("is_current"))
    joined = current_records.alias("e").join(updates_df.alias("u"), key_col, "inner")
    
    # Detect changes (any tracked column different)
    change_condition = None
    for c in track_cols:
        cond = col(f"e.{c}") != col(f"u.{c}")
        change_condition = cond if change_condition is None else (change_condition | cond)
    
    changed = joined.filter(change_condition).select(col(f"e.{key_col}"))
    
    # Expire old records
    expired = existing_df.join(changed, key_col, "left_semi").filter(col("is_current")).withColumn(
        "expiration_date", current_date()
    ).withColumn("is_current", lit(False))
    
    # Keep unchanged records
    unchanged = existing_df.join(changed, key_col, "left_anti")
    
    # New versions from updates
    new_versions = updates_df.join(changed, key_col, "left_semi").select(
        monotonically_increasing_id().alias("customer_key"),
        col(key_col), col("name"), col("email"), col("segment"), col("kyc_status"),
        current_date().alias("effective_date"),
        lit("9999-12-31").cast("date").alias("expiration_date"),
        lit(True).alias("is_current")
    )
    
    return unchanged.union(expired).union(new_versions)

# Apply SCD2
result = apply_scd2(dim_customer_init, updated_customers, "customer_id", ["segment", "kyc_status"])
print(f"After SCD2: {result.count()} records")
result.filter(col("customer_id") == "CUST000001").show()

## 4. Query Patterns

In [ ]:
# Current state
result.filter(col("is_current")).show(5)

# Historical state as-of date
as_of_date = "2025-06-15"
result.filter(
    (col("effective_date") <= as_of_date) & (col("expiration_date") >= as_of_date)
).show(5)

## Practice Exercises
1. Add version_number column to track how many times a customer changed
2. Implement SCD Type 1 (overwrite) for comparison
3. Create a view showing customers with multiple versions

In [ ]:
spark.stop()